In [21]:
import datetime
from dataclasses import dataclass, field, asdict
import json
from zoneinfo import ZoneInfo
from pathlib import Path
from uuid import uuid4, UUID
from typing import Literal

import polars as pl
from loguru import logger

pl.Config(set_tbl_rows=-1, set_tbl_cols=-1)

In [22]:
@dataclass
class DataTracking:
    id: UUID
    column: str
    old_value: any
    new_value: any
    change_type: Literal["create", "update", "delete"]
    origin_source: Path
    new_source: Path
    created_at: datetime.datetime = field(default_factory=datetime.datetime.now)
    updated_at: datetime.datetime | None = None

In [23]:
def read_checkpoint(file_path: str = "checkpoint/checkpoint.json") -> dict | None:
    """Read checkpoint data."""

    with open(file_path, "r") as f:
        try:
            checkpoint = json.load(f)
            return checkpoint if checkpoint else {}

        except json.decoder.JSONDecodeError as e:
            logger.error(e)
            return None

In [24]:
checkpoint = read_checkpoint("/home/user/workspace3/checkpoint/checkpoint.json")

In [25]:
# Load origin data to raw and capture history
brozend_file_path: Path = Path("/home/user/workspace3/data/brozen/data.parquet")

In [26]:
# Get last load data time
last_load_data_time = datetime.datetime.strptime(checkpoint["load_data"]["end_at"], "%Y-%m-%d %H:%M:%S").replace(tzinfo=ZoneInfo("Asia/Ho_Chi_Minh"))
last_load_data_time

datetime.datetime(2026, 7, 28, 18, 9, 18, tzinfo=zoneinfo.ZoneInfo(key='Asia/Ho_Chi_Minh'))

In [27]:
# Load new data from staging
new_source_path = "/home/user/workspace3/data/staging/v2/data.parquet"
df_new = pl.scan_parquet(new_source_path)
df_new.first().collect()

index,id,name,age,gender,address,created_at,updated_at
u32,str,str,i64,str,str,"datetime[μs, Asia/Ho_Chi_Minh]","datetime[μs, Asia/Ho_Chi_Minh]"
1,"""44a6bafe-fa80-4e56-818a-aaf61b…","""user_1_change_all""",21,"""male_change_all""","""address_0_change_all""",2026-07-26 18:09:17.405005 +07,2026-07-28 18:09:17.405203 +07


In [28]:
# Load current data in brozen
current_data_path = "/home/user/workspace3/data/brozen/data.parquet"
df_current = pl.scan_parquet(current_data_path)
df_current.first().collect()

index,id,name,age,gender,address,created_at,updated_at
u32,str,str,i64,str,str,"datetime[μs, Asia/Ho_Chi_Minh]","datetime[μs, Asia/Ho_Chi_Minh]"
1,"""44a6bafe-fa80-4e56-818a-aaf61b…","""user_1""",20,"""male""","""address_0""",2026-07-27 18:09:17.405005 +07,2026-07-27 18:09:17.405203 +07


In [29]:
# Hash row data
import hashlib

def hash_row_data(data) -> str:
    row_str = "|".join(str(value) for value in data)
    return hashlib.sha256(row_str.encode("utf-8")).hexdigest()

# Example:
row_data_current = hash_row_data(df_current.first().collect())
row_data_new = hash_row_data(df_new.first().collect())

print(row_data_current)
print(row_data_new)
print(row_data_current == row_data_new)

67cac7de684e40111d9a5ef286e7d70f0d55d35c8655d013a1cb653f107f16bc
5e9ee8d7be91eced564d8d4bfceef6218ae09bd934ca881b7e970e4742e39b51
False


In [30]:
import hashlib
import polars as pl

def batch_sha256(s: pl.Series) -> pl.Series:
    """Hashes a Polars Series in bulk using Python's hashlib."""
    return pl.Series(
        [hashlib.sha256(val.encode("utf-8")).hexdigest() for val in s]
    )

df_current_hashed = (
    df_current.with_columns(
        pl.all().cast(pl.String).fill_null("")
    )
    .with_columns(
        pl.concat_str(pl.all(), separator="|")
        .map_batches(batch_sha256)
        .alias("hashed_row")
    )
)

df_current_hashed.first().collect()

index,id,name,age,gender,address,created_at,updated_at,hashed_row
str,str,str,str,str,str,str,str,str
"""1""","""44a6bafe-fa80-4e56-818a-aaf61b…","""user_1""","""20""","""male""","""address_0""","""2026-07-27 18:09:17.405005+07:…","""2026-07-27 18:09:17.405203+07:…","""992e810853ddfc80eac69eec8f3f0c…"


In [31]:
df_new_hashed = (
    df_new.with_columns(
        pl.all().cast(pl.String).fill_null("")
    )
    .with_columns(
        pl.concat_str(pl.all(), separator="|")
        .map_batches(batch_sha256)
        .alias("hashed_row")
    )
)

df_new_hashed.first().collect()

index,id,name,age,gender,address,created_at,updated_at,hashed_row
str,str,str,str,str,str,str,str,str
"""1""","""44a6bafe-fa80-4e56-818a-aaf61b…","""user_1_change_all""","""21""","""male_change_all""","""address_0_change_all""","""2026-07-26 18:09:17.405005+07:…","""2026-07-28 18:09:17.405203+07:…","""a8c1566606bd3e1214242eb38662e9…"


In [32]:
# Find changed record
# df_changed = df_new_hashed.join(df_current_hashed, how="anti", on="hashed_row")
# df_changed.select(pl.len()).collect().item()

In [33]:
# Find record new
df_data_new = df_new_hashed.filter(
    ~pl.col("id").is_in(
        df_current_hashed.select("id")
        .collect()
        .to_series()
        .implode()
    )
)
logger.info(f"Data new: {df_data_new.select(pl.len()).collect().item()}")

2026-07-28 11:09:20.564 | INFO     | __main__:<module>:10 - Data new: 10


In [34]:
# Find record deleted
df_data_deleted = df_current_hashed.filter(
    ~pl.col("id").is_in(
        df_new_hashed.select("id")
        .collect()
        .to_series()
        .implode()
    ))
logger.info(f"Data deleted: {df_data_deleted.select(pl.len()).collect().item()}")

2026-07-28 11:09:20.641 | INFO     | __main__:<module>:9 - Data deleted: 10


In [35]:
# Find record not change
df_data_not_change = df_current_hashed.filter(
    pl.col("hashed_row").is_in(
        df_new_hashed.select("hashed_row")
        .collect()
        .to_series()
        .implode()
    )
)
logger.info(f"Data not change: {df_data_not_change.select(pl.len()).collect().item()}")

2026-07-28 11:09:20.709 | INFO     | __main__:<module>:10 - Data not change: 20


In [36]:
# Find data change
exclude_ids = df_data_new.select("id").collect().to_series().to_list() + df_data_not_change.select("id").collect().to_series().to_list()
len(exclude_ids)

df_new_hashed_change = df_new_hashed.filter(
    ~pl.col("id").is_in(exclude_ids)
)
df_new_hashed_change.select(pl.len()).collect().item()

70

In [37]:
exclude_ids = df_data_deleted.select("id").collect().to_series().to_list() + df_data_not_change.select("id").collect().to_series().to_list()
len(exclude_ids)
df_current_hashed_change = df_current_hashed.filter(
    ~pl.col("id").is_in(exclude_ids)
)
df_current_hashed_change.select(pl.len()).collect().item()

70

In [38]:
business_cols = [str(key) for key in df_current.collect_schema() if key not in ("id", "index")]
business_cols

['name', 'age', 'gender', 'address', 'created_at', 'updated_at']

In [39]:
# if df_new_hashed_change.select(pl.len()).collect().item() > 0:
change_log = []
for col in business_cols:
    merged = (
                df_current_hashed_change.select(["id", col]).rename({col: "old_value"})
                .join(df_new_hashed_change.select(["id", col]).rename({col: "new_value"}), on="id")
                .filter(pl.col("old_value") != pl.col("new_value"))
                .collect()
            )
    if len(merged) > 0:
        change_log.append(merged.with_columns(pl.lit(col).alias("column")))
change_log

[shape: (20, 4)
 ┌─────────────────────────────────┬───────────┬─────────────────────┬────────┐
 │ id                              ┆ old_value ┆ new_value           ┆ column │
 │ ---                             ┆ ---       ┆ ---                 ┆ ---    │
 │ str                             ┆ str       ┆ str                 ┆ str    │
 ╞═════════════════════════════════╪═══════════╪═════════════════════╪════════╡
 │ 44a6bafe-fa80-4e56-818a-aaf61b… ┆ user_1    ┆ user_1_change_all   ┆ name   │
 │ 9f5b1f54-1be6-41a4-99a2-470b17… ┆ user_2    ┆ user_2_change_all   ┆ name   │
 │ 00c62c9d-1d87-4f07-b6a1-e5ca64… ┆ user_3    ┆ user_3_change_all   ┆ name   │
 │ eba44ee1-49ca-476b-bd90-cd4262… ┆ user_4    ┆ user_4_change_all   ┆ name   │
 │ a8db67f8-862b-44a6-a643-f1220b… ┆ user_5    ┆ user_5_change_all   ┆ name   │
 │ b2233fde-1004-4465-b302-f1a22a… ┆ user_6    ┆ user_6_change_all   ┆ name   │
 │ 6ad50c96-3690-47bb-ad54-ccd32e… ┆ user_7    ┆ user_7_change_all   ┆ name   │
 │ 1f64b83d-2e73-4438-b3

In [46]:
pl.concat(change_log).with_columns(
    pl.lit(datetime.datetime.now(tz=ZoneInfo("Asia/Ho_Chi_Minh"))).alias("capture_at")
).write_database(
    table_name="changelog",
    connection="sqlite:////home/user/workspace3/changelog/changelog.db",  # Path to SQLite file
    if_table_exists="append",  # 'fail', 'replace', or 'append'
)

120

In [ ]:
# Delete data
df_current_hashed = df_current_hashed.filter(
    ~pl.col("id").is_in(
        df_data_deleted.select("id")
        .collect()
        .to_series()
        .implode()
    ))

df_current_hashed = df_current_hashed.filter(
    ~pl.col("id")
    .is_in(
        df_current_hashed_change.select("id")
        .collect()
        .to_series()
        .implode()
    )
)

In [ ]:
# Add new data
df_current_hashed = pl.concat([df_current_hashed, df_data_new, df_new_hashed_change])

In [ ]:
df_current_hashed.select(pl.len()).collect().item()

100

In [ ]:
df_current_hashed.collect()

index,id,name,age,gender,address,created_at,updated_at,hashed_row
str,str,str,str,str,str,str,str,str
"""72""","""6bafdc33-0a08-4f8b-b065-a65836…","""user_72""","""42""","""male""","""address_71""","""2026-07-27 15:07:33.452448+07:…","""2026-07-27 15:07:33.452583+07:…","""a168b922fdb4b78277834922f511d3…"
"""73""","""8101f717-b7d9-4b32-824f-9a2d6e…","""user_73""","""20""","""female""","""address_72""","""2026-07-27 15:07:33.452449+07:…","""2026-07-27 15:07:33.452584+07:…","""887a3cb250c84c216f07e54087b580…"
"""74""","""0949b06b-63bd-4bfa-b42b-0746f7…","""user_74""","""58""","""male""","""address_73""","""2026-07-27 15:07:33.452450+07:…","""2026-07-27 15:07:33.452586+07:…","""32982d271c35ae32e2a4c124fb47de…"
"""75""","""11ae7024-cd58-4c6d-bf4d-b690a2…","""user_75""","""24""","""other""","""address_74""","""2026-07-27 15:07:33.452451+07:…","""2026-07-27 15:07:33.452587+07:…","""8c6c71a1152c2390c2bf28c3dfea03…"
"""76""","""b2c121df-4d3c-41e7-9d36-e73103…","""user_76""","""19""","""female""","""address_75""","""2026-07-27 15:07:33.452453+07:…","""2026-07-27 15:07:33.452588+07:…","""ff9e5f2d5b5f1a61b729a826f53386…"
"""77""","""4fd7e88e-779b-44fb-a410-abde6c…","""user_77""","""21""","""male""","""address_76""","""2026-07-27 15:07:33.452454+07:…","""2026-07-27 15:07:33.452589+07:…","""44e9490d906e97e425ab4b3d997e2f…"
"""78""","""4d239862-96dd-4edb-b74e-bdaaa9…","""user_78""","""44""","""female""","""address_77""","""2026-07-27 15:07:33.452455+07:…","""2026-07-27 15:07:33.452591+07:…","""735e8e2533a1328d58c6d52df1ad20…"
"""79""","""8c679dc1-11f4-422d-8c7a-39023c…","""user_79""","""42""","""other""","""address_78""","""2026-07-27 15:07:33.452456+07:…","""2026-07-27 15:07:33.452592+07:…","""373c96b22c74c6e6f027d503efb2db…"
"""80""","""159da821-2b10-4384-a05c-5cda1a…","""user_80""","""33""","""female""","""address_79""","""2026-07-27 15:07:33.452458+07:…","""2026-07-27 15:07:33.452593+07:…","""c1fc42e78cf275a03301d859976f96…"


In [ ]:
df_current_hashed.drop("hashed_row").sink_parquet("/home/user/workspace3/data/brozen/data.parquet")

if updated at and created at not change, but any column change inside it? how to track, if cdc just check based on updated at? missing many case: 

df_update_all, -> can track

df_update_name, -> not track

df_update_age, -> not track

df_update_gender, -> not track

df_update_address, -> not track

df_update_created_at, -> not track

df_update_updated_at, -> track

df_new, -> track